<a href="https://colab.research.google.com/github/kuds/rl-doom/blob/main/notebooks/02_dqn_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 — DQN Training (Stable-Baselines3)

Train a **DQN** agent (via `stable_baselines3.DQN`, `CnnPolicy`) on three
ViZDoom scenarios: *Basic*, *Deadly Corridor*, and *Defend the Center*.

**Design notes.** Each scenario trains end-to-end and writes its full
artifact bundle (learning curves, eval curve, video, checkpoint, stage
summary) to disk **before** the next scenario starts, so a Colab timeout
or an error on scenario N+1 still leaves scenario N fully shippable.

**Colab baseline.** Defaults target an **L4 GPU + high-memory runtime**:
single `DummyVecEnv` worker (SB3's DQN only supports `n_envs=1`),
`buffer_size=100_000` frames, `batch_size=64`, `learning_starts=10_000`.

## 1. Setup

In [ ]:
# --- Colab Setup ---
# Uncomment the block below when running on Google Colab
import subprocess, os
if not os.path.exists("/content/rl-doom"):
    subprocess.run(["git", "clone", "https://github.com/kuds/rl-doom.git", "/content/rl-doom"], check=True)
os.chdir("/content/rl-doom/notebooks")
subprocess.run(["pip", "install", "-q", "-e", "/content/rl-doom[notebooks]"], check=True)

import sys
sys.path.insert(0, os.path.abspath("../src"))

import numpy as np
import torch

from rl_doom.paths import new_run_dir, write_config
from rl_doom.sb3_utils import gpu_info, train_sb3

torch.backends.cudnn.benchmark = True
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU_INFO = gpu_info()
print(f"Using device: {DEVICE}")
for k, v in GPU_INFO.items():
    print(f"  {k}: {v}")

In [ ]:
# Google Drive integration for persistent storage on Colab
# Uncomment the block below when running on Google Colab
# ---
import shutil
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = "/content/drive/MyDrive/Finding Theta/rl-doom"
os.makedirs(DRIVE_ROOT, exist_ok=True)
for subdir in ["training_jobs", "analysis"]:
    drive_dir = f"{DRIVE_ROOT}/{subdir}"
    local_dir = os.path.abspath(f"../{subdir}")
    os.makedirs(drive_dir, exist_ok=True)
    if os.path.islink(local_dir):
        os.remove(local_dir)
    if os.path.isdir(local_dir):
        for f in os.listdir(local_dir):
            src = os.path.join(local_dir, f)
            dst = os.path.join(drive_dir, f)
            if not os.path.exists(dst):
                shutil.move(src, dst)
        shutil.rmtree(local_dir)
    os.symlink(drive_dir, local_dir)
print(f"Google Drive mounted. Artifacts will persist at: {DRIVE_ROOT}")
# ---

## 2. DQN hyperparameters (L4 + high-memory defaults)

In [ ]:
SEED = 42

# Shared hyperparameters for all three scenarios. Keep these aligned with
# ``configs/dqn_*.yaml`` so the standalone configs match the notebook.
DQN_HYPERPARAMS = dict(
    lr=1e-4,
    buffer_size=100_000,
    learning_starts=10_000,
    batch_size=64,
    tau=1.0,                     # hard target update
    gamma=0.99,
    train_freq=4,                # gradient step every 4 env steps
    gradient_steps=1,
    target_update_interval=1_000,
    exploration_fraction=0.3,    # ε anneals over first 30% of training
    eps_start=1.0,
    eps_end=0.05,
    max_grad_norm=10.0,
)

SCENARIOS = [
    dict(name="basic",             total_timesteps=200_000, overrides={}),
    dict(name="deadly_corridor",   total_timesteps=500_000, overrides={"buffer_size": 200_000}),
    dict(name="defend_the_center", total_timesteps=500_000, overrides={"buffer_size": 200_000}),
]
DQN_HYPERPARAMS

## 3. Train each scenario (artifacts written per scenario)

`train_sb3` does the full end-to-end training for one scenario. SB3's DQN
requires `n_envs=1`, so the replay buffer is populated in a single worker.
For each scenario we write the checkpoint, learning curves, eval curve,
gameplay video, and stage summary **before** moving on to the next one.

In [ ]:
from IPython.display import Video, display

results = {}
for spec in SCENARIOS:
    scenario = spec["name"]
    hp = {**DQN_HYPERPARAMS, **spec["overrides"]}
    total_ts = spec["total_timesteps"]

    print("\n" + "=" * 70)
    print(f"[DQN] {scenario}  |  total_timesteps={total_ts:,}")
    print("=" * 70)

    run_dir = new_run_dir(scenario, "dqn", seed=SEED)
    write_config(
        run_dir,
        env=scenario,
        algo="dqn",
        seed=SEED,
        hyperparams={**hp, "total_timesteps": total_ts, "n_envs": 1},
        gpu_setup=GPU_INFO,
    )

    def _show(rd, scenario=scenario):
        vids = sorted((rd / "media").glob(f"dqn_{scenario}.*"))
        if vids:
            print(f"[video] {vids[0]}")
            try:
                display(Video(str(vids[0]), embed=True))
            except Exception as exc:
                print(f"  (inline display skipped: {exc})")

    result = train_sb3(
        algo="dqn",
        scenario=scenario,
        run_dir=run_dir,
        hyperparams=hp,
        seed=SEED,
        total_timesteps=total_ts,
        n_envs=1,                # SB3 DQN is single-env only
        eval_freq=10_000,
        eval_episodes=10,
        checkpoint_freq=50_000,
        record_video=True,
        device=DEVICE,
        on_complete=_show,
    )
    results[scenario] = result
    print(
        f"[done] {scenario}: wall={result['wall_time_seconds']:.1f}s | "
        f"fps={result['fps']:.0f} | eval_mean={result['mean_eval_reward']}"
    )

print("\nAll DQN scenarios complete.")
for s, r in results.items():
    print(f"  - {s}: {r['run_dir']}")

## 4. Cross-scenario eval summary

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

plt.figure(figsize=(10, 5))
for scenario, result in results.items():
    npz_path = Path(result["run_dir"]) / "metrics" / "training.npz"
    if not npz_path.exists():
        continue
    data = np.load(npz_path)
    eval_log = data["eval_rewards"]
    if eval_log.ndim == 2 and eval_log.shape[0] > 0:
        steps = eval_log[:, 0]
        means = eval_log[:, 1]
        stds = eval_log[:, 2]
        plt.plot(steps, means, marker="o", label=scenario)
        plt.fill_between(steps, means - stds, means + stds, alpha=0.2)
plt.xlabel("Environment Steps")
plt.ylabel("Eval Reward")
plt.title("DQN — Cross-scenario eval")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Disconnect runtime

In [ ]:
# Disconnect the Colab runtime at the end of the notebook to save compute.
# No-op when running locally.
try:
    from google.colab import runtime as _colab_runtime
except ImportError:
    pass
else:
    import time
    print("Notebook finished. Disconnecting runtime in 5 seconds...")
    time.sleep(5)
    _colab_runtime.unassign()